<a href="https://colab.research.google.com/github/KristinaA96/_DTSC3020_Fall2025/blob/main/Assignment5_Ch_10%2611.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment5: CRM Cleanup @ **DalaShop**
*Files (Ch.10), Exceptions (Ch.10), Unit Tests (Ch.11), and Regular Expressions*  
.....

**Total: 3 points**  (Two questions, 1.5 pts each)  

> This assignment is scenario-based and aligned with Python Crash Course Ch.10 (files & exceptions), Ch.11 (unit testing with `unittest`), and Regular Expressions.

## Scenario
You are a data intern at an online retailer called **DalaShop**.  
Sales exported a **raw contacts** file from the CRM. It contains customer names, emails, and phone numbers, but the formatting is messy and some emails are invalid.  
Your tasks:

1. **Clean** the contacts (Files + Exceptions + Regex).  
2. **Write unit tests** to make sure your helper functions work correctly and keep working in the future.

## Data file (given by the company): `contacts_raw.txt`
Use this exact sample data (you may extend it for your own testing, but do **not** change it when submitting).  
Run the next cell once to create the file beside your notebook.

In [2]:
# Create the provided company dataset file
with open("contacts_raw.txt", "w", encoding="utf-8") as f:
    f.write('Alice Johnson <alice@example.com> , +1 (469) 555-1234\nBob Roberts <bob[at]example.com> , 972-555-777\nSara M. , sara@mail.co , 214 555 8888\n"Mehdi A." <mehdi.ay@example.org> , (469)555-9999\nDelaram <delaram@example.io>, +1-972-777-2121\nNima <NIMA@example.io> , 972.777.2121\nduplicate <Alice@Example.com> , 469 555 1234')
print("Wrote contacts_raw.txt with sample DalaShop data.")

Wrote contacts_raw.txt with sample DalaShop data.


## Q1 (1.5 pts) — CRM cleanup with Files, Exceptions, and Regex
Implement `q1_crm_cleanup.py` to:

1. **Read** `contacts_raw.txt` using `pathlib` and `with`. If the file is missing, **handle** it gracefully with `try/except FileNotFoundError` (print a friendly message; do not crash).
2. **Validate emails** with a simple regex (`r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"`).  
   - Trim whitespace with `strip()` before checking.  
   - Use **full** matching (not partial).
3. **Normalize phone numbers:** remove all non-digits (e.g., with `re.sub(r"\D", "", raw)`).  
   - If the result has **≥ 10 digits**, keep the **last 10 digits**.  
   - Otherwise, return an **empty string** (`""`).
4. **Filter rows:** keep **only** rows with a valid email.
5. **Deduplicate:** remove duplicates by **email** using **case-insensitive** comparison (e.g., `email.casefold()`). **Keep the first occurrence** and drop later duplicates.
6. **Output CSV:** write to `contacts_clean.csv` with **columns exactly** `name,email,phone` (UTF-8).  
7. **Preserve input order:** the order of rows in `contacts_clean.csv` must match the **first appearance** order from the input file. **Do not sort** the rows.

**Grading rubric (1.5 pts):**
- (0.4) File read/write via `pathlib` + graceful `FileNotFoundError` handling  
- (0.5) Correct email regex validation + filtering  
- (0.4) Phone normalization + case-insensitive de-dup (keep first)  
- (0.2) Clean code, clear names, minimal docstrings/comments

In [5]:
q1_crm_cleanup_path = "q1_crm_cleanup.py"
contacts_raw_filename = "contacts_raw.txt"
contacts_clean_filename = "contacts_clean.csv"

In [11]:
import re
import csv
from pathlib import Path

def is_valid_email(email):
    """Validates an email address using a simple regex."""
    if not isinstance(email, str):
        return False
    email = email.strip()
    # The regex provided in the assignment description
    regex = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    return re.fullmatch(regex, email) is not None

def normalize_phone(phone):
    """Normalizes a phone number by removing non-digits and keeping the last 10 digits."""
    if not isinstance(phone, str):
        return ""
    digits = re.sub(r"\D", "", phone)
    if len(digits) >= 10:
        return digits[-10:]
    else:
        return ""

def clean_crm_data(raw_filename, clean_filename):
    """
    Reads raw contact data, cleans it, and writes to a clean CSV file.
    Handles FileNotFoundError gracefully.
    """
    raw_filepath = Path(raw_filename)
    cleaned_contacts = []
    seen_emails = set()

    try:
        with raw_filepath.open("r", encoding="utf-8") as f:
            # Assuming each line is a contact entry
            for line in f:
                # Simple parsing: Split by comma. This might need refinement
                # based on actual data structure, but for the sample data it works.
                parts = [part.strip() for part in line.split(',')]
                name, email, phone = None, None, None

                # Attempt to extract name, email, and phone based on common patterns
                # This is a simplified approach based on the sample data structure
                if len(parts) >= 3:
                    # Example: "Sara M. , sara@mail.co , 214 555 8888"
                    name = parts[0]
                    email = parts[1]
                    phone = parts[2]
                elif len(parts) == 2:
                    # Example: "Alice Johnson <alice@example.com> , +1 (469) 555-1234"
                    # This format requires more sophisticated parsing to separate name and email
                    match = re.match(r'(.*?)<(.*?)>', parts[0])
                    if match:
                        name = match.group(1).strip()
                        email = match.group(2).strip()
                        phone = parts[1]
                    else:
                         # Handle cases like "Nima <NIMA@example.io> , 972.777.2121"
                         # Attempt to split by space if no <> found in the first part
                         name_email_parts = parts[0].split(maxsplit=1)
                         if len(name_email_parts) == 2:
                             name = name_email_parts[0]
                             email = name_email_parts[1]
                             phone = parts[1]
                         elif len(name_email_parts) == 1:
                            # Handle cases with only a name or email in the first part
                            if is_valid_email(name_email_parts[0]):
                                email = name_email_parts[0]
                            else:
                                name = name_email_parts[0]
                            phone = parts[1]


                if email:
                    email = email.strip() # Ensure email is stripped before validation

                # Validate email
                if is_valid_email(email):
                    normalized_phone = normalize_phone(phone)
                    casefolded_email = email.casefold()

                    # Deduplicate based on case-insensitive email
                    if casefolded_email not in seen_emails:
                        seen_emails.add(casefolded_email)
                        # Simple name extraction: if name is None, try to derive from email or first part
                        if name is None or name == '':
                             if email:
                                 name = email.split('@')[0] # Basic extraction before @
                             elif parts and parts[0]:
                                 name = parts[0].strip() # Use the first part if name is empty

                        cleaned_contacts.append({
                            'name': name if name is not None else '', # Ensure name is not None
                            'email': email,
                            'phone': normalized_phone
                        })

    except FileNotFoundError:
        print(f"Error: The file '{raw_filename}' was not found.")
        return

    # Write to CSV
    clean_filepath = Path(clean_filename)
    if cleaned_contacts: # Only write if there are contacts to write
        with clean_filepath.open("w", encoding="utf-8", newline='') as csvfile:
            fieldnames = ['name', 'email', 'phone']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

            writer.writeheader()
            for contact in cleaned_contacts:
                writer.writerow(contact)
        print(f"Cleaned data written to '{clean_filename}'.")
    else:
        print("No valid contacts found after cleaning and filtering.")


if __name__ == "__main__":
    contacts_raw_filename = "contacts_raw.txt"
    contacts_clean_filename = "contacts_clean.csv"
    clean_crm_data(contacts_raw_filename, contacts_clean_filename)

Cleaned data written to 'contacts_clean.csv'.


## Q2 (1.5 pts) — Unit testing with `unittest`
Create tests in `test_crm_cleanup.py` that cover at least:

1. **Email validation**: valid/invalid variations.  
2. **Phone normalization**: parentheses, dashes, spaces, country code; too-short cases.  
3. **Parsing**: from a small multi-line string (not from a file), assert the exact structured rows (name/email/phone).  
4. **De-duplication**: demonstrate that a case-variant duplicate email is dropped (first occurrence kept).




In [12]:
%%writefile test_crm_cleanup.py
import unittest
import re
from pathlib import Path
import csv

# Import the functions from your q1_crm_cleanup.py file
# This assumes q1_crm_cleanup.py is in the same directory
from q1_crm_cleanup import is_valid_email, normalize_phone, clean_crm_data

class TestCRMCleanup(unittest.TestCase):

    # Test cases for is_valid_email
    def test_is_valid_email(self):
        self.assertTrue(is_valid_email("alice@example.com"))
        self.assertTrue(is_valid_email(" sara@mail.co ")) # Test stripping whitespace
        self.assertTrue(is_valid_email("mehdi.ay@example.org"))
        self.assertTrue(is_valid_email("NIMA@example.io")) # Test case-insensitivity (validation should pass)
        self.assertFalse(is_valid_email("bob[at]example.com")) # Invalid format
        self.assertFalse(is_valid_email("invalid-email")) # Missing @
        self.assertFalse(is_valid_email("no-domain@")) # Missing domain
        self.assertFalse(is_valid_email("@nodomain.com")) # Missing local part
        self.assertFalse(is_valid_email(None)) # Test None input
        self.assertFalse(is_valid_email("")) # Test empty string

    # Test cases for normalize_phone
    def test_normalize_phone(self):
        self.assertEqual(normalize_phone("+1 (469) 555-1234"), "4695551234")
        self.assertEqual(normalize_phone("972-555-777"), "") # Too short
        self.assertEqual(normalize_phone("(469)555-9999"), "4695559999")
        self.assertEqual(normalize_phone("972.777.2121"), "9727772121")
        self.assertEqual(normalize_phone("1234567890"), "1234567890") # Exactly 10 digits
        self.assertEqual(normalize_phone("1234567890123"), "34567890123") # More than 10 digits, keep last 10
        self.assertEqual(normalize_phone("abc-def-ghi"), "") # No digits
        self.assertEqual(normalize_phone(None), "") # Test None input
        self.assertEqual(normalize_phone(""), "") # Test empty string

    # Add more test methods here for parsing and deduplication later

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

Writing test_crm_cleanup.py


In [15]:
# Run the tests
!python -m unittest test_crm_cleanup.py

.F
FAIL: test_normalize_phone (test_crm_cleanup.TestCRMCleanup.test_normalize_phone)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/test_crm_cleanup.py", line 32, in test_normalize_phone
    self.assertEqual(normalize_phone("1234567890123"), "34567890123") # More than 10 digits, keep last 10
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: '4567890123' != '34567890123'
- 4567890123
+ 34567890123
? +


----------------------------------------------------------------------
Ran 2 tests in 0.001s

FAILED (failures=1)


## Grading rubric (total 3 pts)
- **Q1 (1.5 pts)**  
  - (0.4) File I/O with `pathlib` + graceful `FileNotFoundError` handling  
  - (0.5) Email validation (regex + strip + full match) and filtering  
  - (0.4) Phone normalization and **case-insensitive** de-duplication (keep first)  
  - (0.2) Code clarity (names, minimal docstrings/comments)
- **Q2 (1.5 pts)**  
  - (0.6) Meaningful coverage for email/phone functions (valid & invalid)  
  - (0.6) Parsing & de-dup tests that assert exact expected rows  
  - (0.3) Standard `unittest` structure and readable test names
